Day 10 – Energy & Utilities
Optimizing Renewable Asset Placement Using Spatial Lakehouse (No Tables, Fully In-Memory)
Use Case: Scoring Land Parcels for Solar Farm Installation
🌍 1. Why Spatial Analytics Matters in Clean Energy

Solar and wind site selection is a multi-criteria spatial decision:

How much solar radiation falls here?

How close is the nearest electrical substation?

Is the land legally usable and risk-free?

Does terrain enable economical construction?

The Spatial Lakehouse on Databricks helps planners integrate this messy spatial data and produce suitability rankings efficiently — even at a nationwide scale.

This walkthrough shows how to:

✅ Generate candidate land parcels (polygons)

✅ Load solar heatmap points

✅ Map grid connections (substations)

✅ Join it all in-memory with Databricks Spatial SQL

✅ Produce a ranked suitability index per land parcel

No tables, no DLT, no write-outs. Everything is DataFrame-based.

🟫 Step 1 — Simulated Land Parcels (POLYGON Geometry)

In [0]:
Day 10 – Energy & Utilities
Optimizing Renewable Asset Placement Using Spatial Lakehouse (No Tables, Fully In-Memory)
Use Case: Scoring Land Parcels for Solar Farm Installation
🌍 1. Why Spatial Analytics Matters in Clean Energy

Solar and wind site selection is a multi-criteria spatial decision:

How much solar radiation falls here?

How close is the nearest electrical substation?

Is the land legally usable and risk-free?

Does terrain enable economical construction?

The Spatial Lakehouse on Databricks helps planners integrate this messy spatial data and produce suitability rankings efficiently — even at a nationwide scale.

This walkthrough shows how to:

✅ Generate candidate land parcels (polygons)

✅ Load solar heatmap points

✅ Map grid connections (substations)

✅ Join it all in-memory with Databricks Spatial SQL

✅ Produce a ranked suitability index per land parcel

No tables, no DLT, no write-outs. Everything is DataFrame-based.

🟫 Step 1 — Simulated Land Parcels (POLYGON Geometry)

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import expr, col
import random

# Simulated land parcels (simple square polygons near Las Vegas)
parcels = [
    Row(parcel_id=f"P_{i}", 
        name=f"Parcel_{i}",
        wkt=f"POLYGON(({lng} {lat}, {lng+0.01} {lat}, {lng+0.01} {lat+0.01}, {lng} {lat+0.01}, {lng} {lat}))"
    )
    for i, (lat, lng) in enumerate([
        (36.0, -115.0), (36.02, -115.04), (36.04, -115.08), (36.06, -115.11)
    ])
]

df_parcels = spark.createDataFrame(parcels) \
    .withColumn("geometry", expr("ST_GEOMFROMWKT(wkt)"))

display(df_parcels)


### Step 2 — Solar Radiation Sampling Points

In [0]:
solar_points = [
    Row(
        lat=36 + random.random()*0.1,
        lon=-115 + random.random()*0.1,
        solar_kwm2=round(random.uniform(4, 7), 2)  # kw/m²/day
    )
    for _ in range(100)
]

df_solar = spark.createDataFrame(solar_points) \
    .withColumn("geo_point", expr("ST_POINT(lon, lat)"))

display(df_solar)


### Step 3 — Grid Connection Points (Substations)

In [0]:
grid_data = [
    Row(name="Substation_1", lat=36.01, lon=-115.03),
    Row(name="Substation_2", lat=36.055, lon=-115.085)
]

df_grid = spark.createDataFrame(grid_data) \
    .withColumn("geo_point", expr("ST_POINT(lon, lat)"))

display(df_grid)


### Step 4 — Renewable Suitability Scoring (No Table Writes)
### ✅ 4.1 — Compute Average Solar Score per Parcel

In [0]:
df_solar.createOrReplaceTempView("solar")
df_parcels.createOrReplaceTempView("parcels")

avg_solar = spark.sql("""
SELECT
  p.parcel_id,
  AVG(s.solar_kwm2) AS avg_solar_kwm2
FROM parcels p
JOIN solar s ON ST_CONTAINS(p.geometry, s.geo_point)
GROUP BY p.parcel_id
""")
display(avg_solar)


4.2 — Compute Distance to Nearest Substation

In [0]:
df_grid.createOrReplaceTempView("grid")

nearest_grid = spark.sql("""
SELECT
  p.parcel_id,
  MIN(ST_DISTANCE(p.geometry, g.geo_point)) AS min_dist_meters
FROM parcels p
CROSS JOIN grid g
GROUP BY p.parcel_id
""")
display(nearest_grid)


### 4.3 — Final Suitability Index (Join + Weighted Formula)

In [0]:
avg_solar.createOrReplaceTempView("solar_scored")
nearest_grid.createOrReplaceTempView("grid_distances")

suitability = spark.sql("""
SELECT
  s.parcel_id,
  s.avg_solar_kwm2,
  g.min_dist_meters,
  -- Scoring model (higher solar, closer to grid = better)
  ROUND(
      (s.avg_solar_kwm2 * 0.6) +
      (10000 / (g.min_dist_meters + 1)) * 0.4, 3
  ) AS suitability_score
FROM solar_scored s
JOIN grid_distances g ON s.parcel_id = g.parcel_id
ORDER BY suitability_score DESC
""")

display(suitability)


You now have a ranked list of renewable development candidates, with meaningful spatial metrics — all in-memory.

### 🗺️ Optional — Visualize Parcels + Score Boundaries on Map (Folium)

In [0]:
%pip install folium
%pip install shapely

In [0]:
import folium
from shapely import wkt
from shapely.geometry import mapping
import json

# Initialize the map
m = folium.Map(location=[36.05, -115.05], zoom_start=11)

# Convert to Pandas and iterate through rows
parcel_pd = df_parcels.toPandas()

for _, row in parcel_pd.iterrows():
    polygon_geom = wkt.loads(row["wkt"])  # Convert WKT to Shapely geometry
    geojson = json.dumps(mapping(polygon_geom))  # Convert to GeoJSON
    folium.GeoJson(
        geojson,
        name=row["name"],
        tooltip=row["name"]
    ).add_to(m)

m



Outcome	Business Value
Rank 100s–100Ks land parcels in minutes	Faster site acquisition
Combine solar yield + infrastructure cost	True feasibility analysis
Native geospatial engine (no GIS servers)	Simplified workflow
Scalable across multiple regions/countries	Ideal for national str


Why This Works

shapely.wkt.loads() reads the WKT string into a geometry object

mapping(...) converts it into a GeoJSON-compatible dict

folium.GeoJson can accept that format (not raw WKT)

Styling for Better Visualization

In [0]:
folium.GeoJson(
    geojson,
    tooltip=row["name"],
    style_function=lambda x: {
        "fillColor": "#3186cc",
        "color": "#3186cc",
        "weight": 2,
        "fillOpacity": 0.2
    }
).add_to(m)
